In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [8]:
df = pd.read_csv("/kaggle/input/bank-customer-churn-dataset/Bank Customer Churn Prediction.csv")
df.head()


,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [9]:
df.shape

(10000, 12)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customer_id       10000 non-null  int64  
 1   credit_score      10000 non-null  int64  
 2   country           10000 non-null  object 
 3   gender            10000 non-null  object 
 4   age               10000 non-null  int64  
 5   tenure            10000 non-null  int64  
 6   balance           10000 non-null  float64
 7   products_number   10000 non-null  int64  
 8   credit_card       10000 non-null  int64  
 9   active_member     10000 non-null  int64  
 10  estimated_salary  10000 non-null  float64
 11  churn             10000 non-null  int64  
dtypes: float64(2), int64(8), object(2)
memory usage: 937.6+ KB


In [11]:
df.describe()

,customer_id,credit_score,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
count,1.000000e+04,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
mean,1.569094e+07,650.528800,38.921800,5.012800,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700
std,7.193619e+04,96.653299,10.487806,2.892174,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769
min,1.556570e+07,350.000000,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000
25%,1.562853e+07,584.000000,32.000000,3.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000
50%,1.569074e+07,652.000000,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000
75%,1.575323e+07,718.000000,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000
max,1.581569e+07,850.000000,92.000000,10.000000,250898.090000,4.000000,1.00000,1.000000,199992.480000,1.000000


In [12]:
df.isnull().sum()

customer_id         0
credit_score        0
country             0
gender              0
age                 0
tenure              0
balance             0
products_number     0
credit_card         0
active_member       0
estimated_salary    0
churn               0
dtype: int64

In [13]:
df.columns 

Index(['customer_id', 'credit_score', 'country', 'gender', 'age', 'tenure',
       'balance', 'products_number', 'credit_card', 'active_member',
       'estimated_salary', 'churn'],
      dtype='object')

In [14]:
df.drop(columns=["customer_id"], inplace=True)


In [15]:
df["churn"].value_counts()

churn
0    7963
1    2037
Name: count, dtype: int64

In [16]:
X = df.drop("churn", axis=1)
y = df["churn"]

In [17]:
categorical_cols = ["country", "gender"]

numerical_cols = [
    "credit_score",
    "age",
    "tenure",
    "balance",
    "products_number",
    "credit_card",
    "active_member",
    "estimated_salary"
]

In [18]:
df.select_dtypes(include="object").columns


Index(['country', 'gender'], dtype='object')

In [19]:
from sklearn.preprocessing import LabelEncoder


In [20]:
le_country = LabelEncoder()
le_gender = LabelEncoder()


In [21]:
df["country"] = le_country.fit_transform(df["country"])
df["gender"] = le_gender.fit_transform(df["gender"])


In [22]:
df.head()

,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,619,0,0,42,2,0.00,1,1,1,101348.88,1
1,608,2,0,41,1,83807.86,1,0,1,112542.58,0
2,502,0,0,42,8,159660.80,3,1,0,113931.57,1
3,699,0,0,39,1,0.00,2,0,0,93826.63,0
4,850,2,0,43,2,125510.82,1,1,1,79084.10,0


In [23]:
X_encoded = pd.get_dummies(
    df.drop("churn", axis=1),
    drop_first=True
)

y = df["churn"]

from sklearn.model_selection import train_test_split

X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [24]:
from sklearn.ensemble import RandomForestClassifier

rf_2 = RandomForestClassifier(
    n_estimators=700,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=4,
    class_weight="balanced",
    random_state=42
)

rf_2.fit(X_train_e, y_train_e)


RandomForestClassifier(class_weight='balanced', max_depth=15,
                       min_samples_leaf=4, min_samples_split=10,
                       n_estimators=700, random_state=42)

In [25]:
rf2_prob = rf_2.predict_proba(X_test_e)[:, 1]
rf2_pred = rf_2.predict(X_test_e)


In [26]:
from sklearn.metrics import classification_report, roc_auc_score, f1_score

def eval_rf(name, y_test, y_pred, y_prob):
    print(name)
    print(classification_report(y_test, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test, y_prob))
    print("F1-score (churn):", f1_score(y_test, y_pred))
    print("-" * 60)


In [27]:
eval_rf("RF Model 2", y_test_e, rf2_pred, rf2_prob)


RF Model 2
              precision    recall  f1-score   support

           0       0.90      0.91      0.91      1593
           1       0.63      0.61      0.62       407

    accuracy                           0.85      2000
   macro avg       0.77      0.76      0.76      2000
weighted avg       0.85      0.85      0.85      2000

ROC-AUC: 0.8579704511907903
F1-score (churn): 0.6217228464419475
------------------------------------------------------------


In [28]:
X_train_e.columns


Index(['credit_score', 'country', 'gender', 'age', 'tenure', 'balance',
       'products_number', 'credit_card', 'active_member', 'estimated_salary'],
      dtype='object')

In [29]:
bank_sample = pd.DataFrame([{
    "credit_score": 420,
    "country": 0,        # France (label encoded)
    "gender": 0,         # Female
    "age": 52,
    "tenure": 2,
    "balance": 145000,
    "products_number": 1,
    "credit_card": 0,
    "active_member": 0,
    "estimated_salary": 35000
}])


In [30]:
prob = rf_2.predict_proba(bank_sample)[0][1]
pred = int(prob >= 0.45)   # best threshold you selected

print("Churn Probability:", round(prob, 3))
print("Prediction:", "CHURN" if pred==1 else "NO CHURN")


Churn Probability: 0.875
Prediction: CHURN


In [31]:
bank_sample2 = pd.DataFrame([{
    "credit_score": 750,
    "country": 1,
    "gender": 1,
    "age": 30,
    "tenure": 8,
    "balance": 20000,
    "products_number": 3,
    "credit_card": 1,
    "active_member": 1,
    "estimated_salary": 120000
}])

bank_sample2 = bank_sample2[X_train_e.columns]

prob = rf_2.predict_proba(bank_sample2)[0][1]
pred = int(prob >= 0.45)

print("Churn Probability:", round(prob, 3))
print("Prediction:", "CHURN" if pred==1 else "NO CHURN")


Churn Probability: 0.523
Prediction: CHURN


In [32]:
bank_sample_safe = pd.DataFrame([{
    "credit_score": 820,
    "country": 1,
    "gender": 1,
    "age": 45,
    "tenure": 9,
    "balance": 180000,
    "products_number": 1,
    "credit_card": 1,
    "active_member": 1,
    "estimated_salary": 60000
}])

bank_sample_safe = bank_sample_safe[X_train_e.columns]

prob = rf_2.predict_proba(bank_sample_safe)[0][1]
pred = int(prob >= 0.45)

print(prob, pred)


0.3467235351150654 0


In [33]:
import joblib

joblib.dump(rf_2, "bank_churn_model.pkl")



['bank_churn_model.pkl']

In [34]:
joblib.dump(le_country, "bank_country_encoder.pkl")
joblib.dump(le_gender, "bank_gender_encoder.pkl")


['bank_gender_encoder.pkl']

In [35]:
joblib.dump(list(X_train_e.columns), "bank_features.pkl")


['bank_features.pkl']